In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

In [2]:
MODEL_NAME = "fitlemon/bge-m3-ru-ostap"

In [3]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [4]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [6]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [7]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [8]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

Filter:   0%|          | 0/112049 [00:00<?, ? examples/s]

In [9]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [10]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'sentiment': sentiment}

In [11]:
ds_russian = ds_russian.map(calc_sentinent)

Map:   0%|          | 0/111667 [00:00<?, ? examples/s]

In [12]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'sentiment'],
    num_rows: 111667
})

In [13]:
ds_russian_splitted = ds_russian['train'].train_test_split(test_size=0.1, seed=42)

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [16]:
def tokenize(batch):
    return tokenizer(batch['combined_text'], truncation=True, max_length=256)

In [17]:
ds_tok = ds_russian_splitted.map(tokenize, batched=True)

Map:   0%|          | 0/100500 [00:00<?, ? examples/s]

Map:   0%|          | 0/11167 [00:00<?, ? examples/s]